# Semi-real demo (selector)

This notebook runs one of three estimators on the semi-real Job Corps benchmark.

Workflow:
- Choose an algorithm in the **Manual usage** cell at the bottom.
- Set `K_RUNS` and `FIRST_SEED`.
- Run to export CSVs to `KRR_methods/Results/`.

Methods:
- **Ours (tensor-product two-stage)**
- **Plug-in (Nystrom + LOOCV)**
- **Direct regression baseline (T-only Laplace KRR)**


In [2]:
import sys
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Environment + path setup ---
# In Colab, mount Drive and use the repo stored in Google Drive.
# Locally, search upward for the repo root that contains KRR_methods.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = pathlib.Path('/content/drive/MyDrive/Colab Notebooks/CTE_Codes')
except ImportError:
    # Local: find repo root by walking up to a folder containing KRR_methods
    BASE_DIR = pathlib.Path('.').resolve()
    for p in [BASE_DIR, *BASE_DIR.parents]:
        if (p / 'KRR_methods').exists():
            BASE_DIR = p
            break
    else:
        raise RuntimeError('Cannot find repo root containing KRR_methods')

# Make local modules importable (repo root + KRR_methods).
sys.path.append(str(BASE_DIR))
sys.path.append(str(BASE_DIR / 'KRR_methods'))

# Project imports: Job Corps data utilities and estimators.
from KRR_methods.data_jobcorps import make_Xss, gen_semi_y, load_jobcorps_data
from KRR_methods.algorithms.estimators_ours import run_single_ours_semireal
from KRR_methods.algorithms.estimators_plugin import run_single_plugin_semireal
from KRR_methods.algorithms.estimators_direct import run_single_direct_semireal

print(f'Working Directory: {BASE_DIR}')

# Output directory for CSV summaries.
RESULTS_DIR = BASE_DIR / 'KRR_methods' / 'Results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('RESULTS_DIR =', RESULTS_DIR)


Mounted at /content/drive
Working Directory: /content/drive/MyDrive/Colab Notebooks/CTE_Codes
RESULTS_DIR = /content/drive/MyDrive/Colab Notebooks/CTE_Codes/KRR_methods/Results


In [3]:
# --- Data loading ---
# Job Corps empirical data + precomputed semi-synthetic components.
EMP_DIR = BASE_DIR / 'DML_methods' / 'Data_and_Results'
DATA_FILE = EMP_DIR / 'emp_app.csv'
SEMI_SYN_FILE = EMP_DIR / 'semi-syn data grf.csv'   # mu_hat_grf, g_grf
H_STAR_FILE = EMP_DIR / 'h_star_grf_empapp.csv'     # ground-truth h*(t)

# Returns:
# X: covariates, T: treatment "d", MU_HAT: baseline prediction (mu_hat_grf),
# G_FUNC: scale for semi-synthetic noise (g_grf),
# T_GRID/H_STAR_VALS: ground-truth curve on a grid.
X, T, MU_HAT, G_FUNC, T_GRID, H_STAR_VALS = load_jobcorps_data(
    data_file=DATA_FILE,
    semi_syn_file=SEMI_SYN_FILE,
    h_star_file=H_STAR_FILE,
)

# Convert to convenient types.
T = T.reset_index(drop=True) if hasattr(T, 'reset_index') else T
X = X.reset_index(drop=True) if hasattr(X, 'reset_index') else X

T_GRID = np.asarray(T_GRID, dtype=float).ravel()
H_STAR_VALS = np.asarray(H_STAR_VALS, dtype=float).ravel()

print('Data loading complete.')
print(f'n={len(T)}, grid_size={len(T_GRID)}')


Loading empirical data from: emp_app.csv...
Loading semi-synthetic components from: semi-syn data grf.csv...
Loading ground truth h*(t) from: h_star_grf_empapp.csv...
Data loading complete.
n=4024, grid_size=47


In [4]:
# --- Common helpers ---

def save_results_csv(results, prefix=None, include_h_star=True):
    """Save per-seed curves and summary statistics to CSV.

    Args:
        results (dict): Output from run_* functions. Expected keys include
            't_grid', 'curves_mat', 'mean_curve', 'se_curve', 'seeds', and 'run_tag'.
        prefix (str or None): Optional label (kept for compatibility).
        include_h_star (bool): If True, include the ground-truth h*(t) column.

    Returns:
        pathlib.Path: Path to the saved CSV file.
    """
    run_tag = results['run_tag']
    output_path = RESULTS_DIR / f'{run_tag}.csv'

    df_out = pd.DataFrame({'t': results['t_grid']})
    for i, seed in enumerate(results['seeds']):
        df_out[f'h_hat_seed_{seed}'] = results['curves_mat'][i, :]

    df_out['mean_h_hat'] = results['mean_curve']
    df_out['se_h_hat'] = results['se_curve']
    if include_h_star and 'h_star' in results:
        df_out['h_star'] = results['h_star']

    df_out.to_csv(output_path, index=False)
    print(f'✅ Saved: {output_path}')
    return output_path


In [5]:
# --- Ours (tensor-product two-stage) ---
# Kernel hyperparameters for the outcome surface f(X, T).
KERNEL_TYPE_F = 'matern'
KRR_X_LENGTH_SCALE = 13
KRR_X_NU = 0.5
KRR_T_LENGTH_SCALE = 6000
KRR_T_NU = 0.5

# Kernel hyperparameters for h(t) smoothing.
KERNEL_TYPE_H = 'matern'
KRR_H_LENGTH_SCALE = 6000
KRR_H_NU = 1.5

# Ridge penalty grids for the two stages.
C_VAL = 0.1
BETA_GRID = np.array([C_VAL * (2**i) for i in range(0, 9)])
BETA0_F = C_VAL
BETA0_PRIME_F = C_VAL

# Second-stage evaluation grid settings.
SECOND_STAGE_N = 2012
L_VAL = 3000.0
SECOND_STAGE_RANGE = (0.0, L_VAL)

def run_ours(K=100, first_seed=1, prefix='ours_semi-real'):
    """Run K Monte Carlo replications of the proposed estimator.

    Args:
        K (int): Number of Monte Carlo runs.
        first_seed (int): RNG seed for the first run (subsequent runs increment).
        prefix (str): Label used in output filenames.

    Returns:
        dict: Curves, MISE values, and summary statistics.
    """
    mise_list, curves_list, seeds = [], [], []

    for k in range(K):
        seed = first_seed + k
        seeds.append(seed)

        np.random.seed(seed)
        rng_sim = np.random.default_rng(seed)

        # Generate semi-synthetic outcomes: Y_syn = MU_HAT + e * G_FUNC.
        Y_syn = gen_semi_y(MU_HAT, G_FUNC, rng_sim)

        # Shuffle (X, T, Y) consistently for each run.
        n = len(T)
        perm = rng_sim.permutation(n)
        Xs = X.iloc[perm].reset_index(drop=True)
        Ts = T.iloc[perm].reset_index(drop=True)
        Ys = Y_syn[perm]

        # Standardize covariates (min-max scaling per column).
        Xss = make_Xss(Xs)

        print(f'--- Run {k + 1}/{K} (seed={seed}) ---')

        out = run_single_ours_semireal(
            Xss,
            Ts,
            Ys,
            t_grid_original=T_GRID,
            h_star_vals_original=H_STAR_VALS,
            beta_grid=BETA_GRID,
            kernel_type_f=KERNEL_TYPE_F,
            ell_x=KRR_X_LENGTH_SCALE,
            nu_x=KRR_X_NU,
            ell_t=KRR_T_LENGTH_SCALE,
            nu_t=KRR_T_NU,
            beta0_f=BETA0_F,
            beta0_prime_f=BETA0_PRIME_F,
            kernel_type_H=KERNEL_TYPE_H,
            l_H=KRR_H_LENGTH_SCALE,
            nu_H=KRR_H_NU,
            second_stage_range=SECOND_STAGE_RANGE,
            second_stage_n=SECOND_STAGE_N,
        )

        mise = out['mise_ours']
        mise_list.append(mise)
        curves_list.append(out['h_ours'])
        print(f'  MISE: {mise:.6f}')

    mise_arr = np.asarray(mise_list, dtype=float)
    curves_mat = np.vstack(curves_list)

    mean_curve = curves_mat.mean(axis=0)
    std_curve = curves_mat.std(axis=0, ddof=1) if K > 1 else np.zeros_like(mean_curve)
    se_curve = std_curve / np.sqrt(K) if K > 1 else np.zeros_like(mean_curve)

    mean_mise = float(mise_arr.mean()) if K > 0 else float('nan')
    std_mise = float(mise_arr.std(ddof=1)) if K > 1 else 0.0
    se_mise = float(std_mise / np.sqrt(K)) if K > 1 else 0.0
    print(f'✅ Done: mean MISE = {mean_mise:.6f} | SE = {se_mise:.6f}')

    run_tag = f'{prefix}_seeds_{seeds[0]}-{seeds[-1]}'
    return {
        'seeds': seeds,
        't_grid': T_GRID,
        'h_star': H_STAR_VALS,
        'mise_all': mise_arr,
        'mise_mean': mean_mise,
        'mise_se': se_mise,
        'curves_mat': curves_mat,
        'mean_curve': mean_curve,
        'se_curve': se_curve,
        'run_tag': run_tag,
    }


In [6]:
# --- Plug-in (Nystrom + LOOCV) ---
PLUGIN_KERNEL_TYPE_F = 'matern'
PLUGIN_KRR_X_LENGTH_SCALE = 13
PLUGIN_KRR_X_NU = 0.5
PLUGIN_KRR_T_LENGTH_SCALE = 6000
PLUGIN_KRR_T_NU = 0.5
PLUGIN_NYSTROM_M_F = 700

PLUGIN_BETA_MIN = 0.05
PLUGIN_BETA_MAX = 80.0

def run_plugin(K=100, first_seed=1, prefix='plugin_semi-real'):
    """Run K Monte Carlo replications of the plug-in baseline.

    Args:
        K (int): Number of Monte Carlo runs.
        first_seed (int): RNG seed for the first run (subsequent runs increment).
        prefix (str): Label used in output filenames.

    Returns:
        dict: Curves, MISE values, and summary statistics.
    """
    mise_list, curves_list, seeds = [], [], []

    for k in range(K):
        seed = first_seed + k
        seeds.append(seed)

        # Match legacy reproducibility: set global seed per run.
        np.random.seed(seed)

        print(f'--- Run {k + 1}/{K} (seed={seed}) ---')

        rng_sim = np.random.default_rng(seed)
        Y_syn = gen_semi_y(MU_HAT, G_FUNC, rng_sim)

        n = len(T)
        perm = rng_sim.permutation(n)
        Xs = X.iloc[perm].reset_index(drop=True)
        Ts = T.iloc[perm].reset_index(drop=True)
        Ys = Y_syn[perm]

        Xss = make_Xss(Xs)

        out = run_single_plugin_semireal(
            Xss,
            Ts,
            Ys,
            t_grid_original=T_GRID,
            h_star_vals_original=H_STAR_VALS,
            beta_min=PLUGIN_BETA_MIN,
            beta_max=PLUGIN_BETA_MAX,
            kernel_type_f=PLUGIN_KERNEL_TYPE_F,
            ell_x=PLUGIN_KRR_X_LENGTH_SCALE,
            nu_x=PLUGIN_KRR_X_NU,
            ell_t=PLUGIN_KRR_T_LENGTH_SCALE,
            nu_t=PLUGIN_KRR_T_NU,
            m_f=PLUGIN_NYSTROM_M_F,
            verbose=False,
        )

        mise = out['mise_plugin']
        mise_list.append(mise)
        curves_list.append(out['h_plugin'])
        print(f'  MISE: {mise:.6f}')

    mise_arr = np.asarray(mise_list, dtype=float)
    curves_mat = np.vstack(curves_list)

    mean_curve = curves_mat.mean(axis=0)
    std_curve = curves_mat.std(axis=0, ddof=1) if K > 1 else np.zeros_like(mean_curve)
    se_curve = std_curve / np.sqrt(K) if K > 1 else np.zeros_like(mean_curve)

    mean_mise = float(mise_arr.mean()) if K > 0 else float('nan')
    std_mise = float(mise_arr.std(ddof=1)) if K > 1 else 0.0
    se_mise = float(std_mise / np.sqrt(K)) if K > 1 else 0.0
    print(f'✅ Done: mean MISE = {mean_mise:.6f} | SE = {se_mise:.6f}')

    run_tag = f'{prefix}_seeds_{seeds[0]}-{seeds[-1]}'
    return {
        'seeds': seeds,
        't_grid': T_GRID,
        'h_star': H_STAR_VALS,
        'mise_all': mise_arr,
        'mise_mean': mean_mise,
        'mise_se': se_mise,
        'curves_mat': curves_mat,
        'mean_curve': mean_curve,
        'se_curve': se_curve,
        'run_tag': run_tag,
    }


In [7]:
# --- Direct regression baseline (T-only Laplace KRR) ---
DIRECT_ELL_T = 3000.0
DIRECT_NYSTROM_M_T = 700
DIRECT_BETA_MIN = 0.05
DIRECT_BETA_MAX = 80.0

def run_direct(K=100, first_seed=1, prefix='direct_semi-real'):
    """Run K Monte Carlo replications of the T-only Laplace KRR baseline.

    Args:
        K (int): Number of Monte Carlo runs.
        first_seed (int): RNG seed for the first run (subsequent runs increment).
        prefix (str): Label used in output filenames.

    Returns:
        dict: Curves, MISE values, and summary statistics.
    """
    mise_list, curves_list, beta_list, seeds = [], [], [], []

    n = len(T)
    print(f"\nRunning T-only Laplace KRR baseline for K={K} runs ...")
    print(f'    Fixed ell_t={DIRECT_ELL_T}, Nystrom m={DIRECT_NYSTROM_M_T}, beta in [{DIRECT_BETA_MIN}, {DIRECT_BETA_MAX}]')

    for k in range(K):
        seed = first_seed + k
        seeds.append(seed)

        np.random.seed(seed)
        print(f'--- Run {k + 1}/{K} (seed={seed}) ---')

        rng_sim = np.random.default_rng(seed)
        Y_syn = gen_semi_y(MU_HAT, G_FUNC, rng_sim)

        perm = rng_sim.permutation(n)
        Ts = T.iloc[perm].reset_index(drop=True) if hasattr(T, 'iloc') else np.asarray(T)[perm]
        Ys = Y_syn[perm]

        h_hat_grid, beta_star, mise = run_single_direct_semireal(
            T_train=np.asarray(Ts, dtype=float),
            Y_train=np.asarray(Ys, dtype=float),
            t_grid=T_GRID,
            h_star_vals=H_STAR_VALS,
            ell_t=DIRECT_ELL_T,
            m_t=DIRECT_NYSTROM_M_T,
            beta_min=DIRECT_BETA_MIN,
            beta_max=DIRECT_BETA_MAX,
            seed_landmarks=0,
        )

        mise_list.append(mise)
        curves_list.append(h_hat_grid)
        beta_list.append(beta_star)
        print(f'  MISE: {mise:.6f}')

    mise_arr = np.asarray(mise_list, dtype=float)
    curves_mat = np.vstack(curves_list)

    mean_curve = curves_mat.mean(axis=0)
    std_curve = curves_mat.std(axis=0, ddof=1) if K > 1 else np.zeros_like(mean_curve)
    se_curve = std_curve / np.sqrt(K) if K > 1 else np.zeros_like(mean_curve)

    beta_arr = np.asarray(beta_list, dtype=float)
    beta_mean = float(beta_arr.mean())
    beta_std = float(beta_arr.std(ddof=1)) if K > 1 else 0.0
    beta_se = float(beta_std / np.sqrt(K)) if K > 1 else 0.0

    mean_mise = float(mise_arr.mean()) if K > 0 else float('nan')
    std_mise = float(mise_arr.std(ddof=1)) if K > 1 else 0.0
    se_mise = float(std_mise / np.sqrt(K)) if K > 1 else 0.0
    print(f'✅ Done: mean MISE = {mean_mise:.6f} | SE = {se_mise:.6f}')

    run_tag = f'{prefix}_seeds_{seeds[0]}-{seeds[-1]}'
    return {
        'seeds': seeds,
        't_grid': T_GRID,
        'h_star': H_STAR_VALS,
        'mise_all': mise_arr,
        'mise_mean': mean_mise,
        'mise_se': se_mise,
        'curves_mat': curves_mat,
        'mean_curve': mean_curve,
        'se_curve': se_curve,
        'beta_selected_all': beta_arr,
        'beta_selected_mean': beta_mean,
        'beta_selected_se': beta_se,
        'run_tag': run_tag,
    }


In [8]:
# --- Selector (no widgets) ---

def dispatch_run(algo, K, first_seed, save_csv=True):
    """Dispatch to a specific estimator by name.

    Args:
        algo (str): One of "ours", "plugin", or "direct" (case-insensitive).
        K (int): Number of Monte Carlo runs.
        first_seed (int): RNG seed for the first run.
        save_csv (bool): If True, save per-seed curves and summaries to CSV.

    Returns:
        dict: Results dictionary returned by the selected run_* function.
    """
    algo = str(algo).lower().strip()
    if algo in ['ours', 'joint-ours', 'joint_ours', 'ours (joint)']:
        results = run_ours(K=K, first_seed=first_seed)
    elif algo in ['plugin', 'plug-in', 'plug_in']:
        results = run_plugin(K=K, first_seed=first_seed)
    elif algo in ['direct', 'direct-regression', 'direct regression', 't-only', 't_only']:
        results = run_direct(K=K, first_seed=first_seed)
    else:
        raise ValueError(f'Unknown algo: {algo}')

    if save_csv:
        save_results_csv(results, prefix=algo, include_h_star=True)

    return results


In [9]:
# --- Manual usage (no widgets) ---
ALGO = 'ours'     # 'ours' | 'plugin' | 'direct'
K_RUNS = 100        # Number of Monte Carlo runs
FIRST_SEED = 1      # RNG seed for the first run

results = dispatch_run(
    algo=ALGO,
    K=K_RUNS,
    first_seed=FIRST_SEED,
    save_csv=True,
)
print('run_tag =', results['run_tag'])


--- Run 1/100 (seed=1) ---
  MISE: 1.994353
--- Run 2/100 (seed=2) ---
  MISE: 0.249440
--- Run 3/100 (seed=3) ---
  MISE: 1.835164
--- Run 4/100 (seed=4) ---
  MISE: 0.607466
--- Run 5/100 (seed=5) ---
  MISE: 0.160391
--- Run 6/100 (seed=6) ---
  MISE: 0.791055
--- Run 7/100 (seed=7) ---
  MISE: 1.143464
--- Run 8/100 (seed=8) ---
  MISE: 1.010764
--- Run 9/100 (seed=9) ---
  MISE: 2.392028
--- Run 10/100 (seed=10) ---
  MISE: 3.122674
--- Run 11/100 (seed=11) ---
  MISE: 4.026220
--- Run 12/100 (seed=12) ---
  MISE: 0.866902
--- Run 13/100 (seed=13) ---
  MISE: 1.589211
--- Run 14/100 (seed=14) ---
  MISE: 2.297269
--- Run 15/100 (seed=15) ---
  MISE: 1.420445
--- Run 16/100 (seed=16) ---
  MISE: 2.070913
--- Run 17/100 (seed=17) ---
  MISE: 0.172606
--- Run 18/100 (seed=18) ---
  MISE: 0.400014
--- Run 19/100 (seed=19) ---
  MISE: 1.793316
--- Run 20/100 (seed=20) ---
  MISE: 1.088182
--- Run 21/100 (seed=21) ---
  MISE: 2.680156
--- Run 22/100 (seed=22) ---
  MISE: 1.089628
--- Ru

/tmp/ipykernel_2666/2330362695.py:20: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out[f'h_hat_seed_{seed}'] = results['curves_mat'][i, :]
/tmp/ipykernel_2666/2330362695.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_out['mean_h_hat'] = results['mean_curve']
/tmp/ipykernel_2666/2330362695.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get 

✅ Saved: /content/drive/MyDrive/Colab Notebooks/CTE_Codes/KRR_methods/Results/ours_semi-real_seeds_1-100.csv
run_tag = ours_semi-real_seeds_1-100
